In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone per rules
tz = pytz.timezone('Asia/Shanghai')

# --- Helper to compute summary for a dataframe given the KPI column name ---
def compute_summary(df, kpi_col, top_n=50):
    # parse timestamp to timezone-aware datetime if present
    if 'timestamp' in df.columns:
        try:
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        except Exception:
            # fallback: just localize as Asia/Shanghai
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    # ensure value column exists
    if 'value' not in df.columns:
        raise ValueError("Input dataframe missing 'value' column")
    # group and aggregate
    grp = df.groupby(['cmdb_id', kpi_col])['value']
    summary = grp.agg(
        count='count',
        min_value='min',
        p50=lambda x: np.percentile(x.dropna(), 50) if len(x.dropna())>0 else np.nan,
        p95=lambda x: np.percentile(x.dropna(), 95) if len(x.dropna())>0 else np.nan,
        max_value='max'
    ).reset_index()
    # normalize column names to (component, kpi_name, count, min, p50, p95, max)
    summary = summary.rename(columns={
        'cmdb_id': 'component',
        kpi_col: 'kpi_name',
        'min_value': 'min',
        'p50': 'median_p50',
        'p95': 'p95',
        'max_value': 'max'
    })
    # sort and limit
    summary = summary.sort_values('count', ascending=False).head(top_n)
    # round numeric columns for compactness
    num_cols = ['min', 'median_p50', 'p95', 'max']
    for c in num_cols:
        if c in summary.columns:
            summary[c] = summary[c].round(6)
    return summary

# --- Load files and compute summaries ---
# metric.csv
df = pd.read_csv('metric.csv')
metric_summary = compute_summary(df, 'kpi_name', top_n=50)

# trace.csv
df = pd.read_csv('trace.csv')
trace_summary = compute_summary(df, 'trace_name', top_n=50)

# log.csv
df = pd.read_csv('log.csv')
log_summary = compute_summary(df, 'log_name', top_n=50)

# Display compact summaries (up to 50 rows each)
metric_summary, trace_summary, log_summary

```
Out[1]:
```
```python
# Summarize the previously computed summaries (metric_summary, trace_summary, log_summary)
# Produce a compact plain-English summary string and display it.

def compact_summary(df, title):
    n = len(df)
    cnt_max = int(df['count'].max()) if n>0 else 0
    cnt_min = int(df['count'].min()) if n>0 else 0
    top_comps = df['component'].value_counts().head(5)
    top_comps_lines = "; ".join([f"{comp}({c})" for comp,c in top_comps.items()]) if not top_comps.empty else "none"
    # example top 3 rows as concise lines
    examples = []
    for _, r in df.head(3).iterrows():
        examples.append(
            f"{r['component']} | {r['kpi_name']} | count={int(r['count'])} | min={r['min']:.6g} | p50={r['median_p50']:.6g} | p95={r['p95']:.6g} | max={r['max']:.6g}"
        )
    examples_text = "  " + "\n  ".join(examples) if examples else "  (none)"
    return (
        f"{title} (showing up to 50 groups):\n"
        f"- Displayed groups: {n}\n"
        f"- Count range (displayed): {cnt_min} … {cnt_max}\n"
        f"- Top components in displayed rows (component(count)): {top_comps_lines}\n"
        f"- Example top rows:\n{examples_text}\n"
    )

summary_parts = [
    compact_summary(metric_summary, "Metric CSV summary"),
    compact_summary(trace_summary, "Trace CSV summary"),
    compact_summary(log_summary,   "Log CSV summary")
]

summary = "\n".join(summary_parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     component                                           kpi_name  count           min    median_p50           p95           max
1660  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1662  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1603  apache02                            OSLinux-CPU_CPU_CPULoad     31  0.000000e+00  3.000000e-02  1.000000e-01  1.300000e-01
1604  apache02                         OSLinux-CPU_CPU_CPUSysTime     31  2.327000e-01  3.492000e-01  4.695790e-01  4.737370e-01
1605  apache02                        OSLinux-CPU_CPU_CPUUserTime     31  1.330000e-01  1.870090e-01  2.410670e-01  2.492940e-01
...        ...                                                ...    ...           ...           ...           ...           ...
1387  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  3.303510e+05  3.303510e+05  3.303510e+05  3.303510e+05
1388  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  2.925872e+08  3.012215e+08  3.103926e+08  3.118546e+08
1389  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  4.147730e+05  4.207890e+05  4.266250e+05  4.272800e+05
1390  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1391  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 7 columns],     component                           kpi_name  count         min  median_p50         p95         max
263  dockerB2            trace.to_MG02.row_count     30   30.000000  125.000000  176.300000  223.000000
262  dockerB2         trace.to_MG02.duration_p95     30    0.161000    5.234350   13.590060   34.315400
261  dockerB2        trace.to_MG02.duration_mean     30    0.049382    2.132534    6.776940    7.259479
260  dockerB2            trace.to_MG01.row_count     30   37.000000  118.500000  199.950000  227.000000
3        IG01           trace.self.duration_mean     30    0.413800    1.426528    3.244914    3.352077
..        ...                                ...    ...         ...         ...         ...         ...
46       MG01   trace.from_Tomcat04.duration_p95     30    1.337000    5.762450   23.587880   35.389600
47       MG01      trace.from_Tomcat04.row_count     30  165.000000  209.500000  259.850000  279.000000
48       MG01  trace.from_dockerA1.duration_mean     30    0.045613    1.807452    7.953240    8.587857
49       MG01   trace.from_dockerA1.duration_p95     30    0.080500    4.245550   14.769875   36.931000
50       MG01      trace.from_dockerA1.row_count     30    7.000000   32.500000   55.150000   64.000000

[50 rows x 7 columns],    component          kpi_name  count    min  median_p50       p95      max
9   Tomcat02     log.row_count     31  12.00     415.000  492.5000   521.00
7   Tomcat02   log.error_count     31   0.00       0.000    0.0000     0.00
6   Tomcat01     log.row_count     31   9.00     411.000  523.0000   537.00
4   Tomcat01   log.error_count     31   0.00       0.000    0.0000     0.00
18  apache02   log.error_count     31   0.00       0.000    0.0000     0.00
17  apache01     log.row_count     31  13.00     835.000  993.5000  1033.00
19  apache02     log.row_count     31  11.00     846.000  983.0000  1040.00
16  apache01   log.error_count     31   0.00       0.000    0.0000     0.00
10  Tomcat03   log.error_count     31   0.00       0.000    0.0000     0.00
15  Tomcat04     log.row_count     31  10.00     408.000  515.0000   533.00
13  Tomcat04   log.error_count     31   0.00       0.000    0.0000     0.00
12  Tomcat03     log.row_count     31   9.00     425.000  493.5000   508.00
8   Tomcat02  log.gc_pause_max     30   0.02       0.075    0.0955     0.10
0    Mysql01   log.error_count     25   0.00       0.000    0.0000     0.00
1    Mysql01     log.row_count     25   1.00       5.000   14.8000    17.00
5   Tomcat01  log.gc_pause_max     23   0.02       0.070    0.1180     0.13
11  Tomcat03  log.gc_pause_max     23   0.03       0.060    0.0890     0.12
14  Tomcat04  log.gc_pause_max     22   0.02       0.070    0.0900     0.13
3    Mysql02     log.row_count      2   1.00       1.000    1.0000     1.00
2    Mysql02   log.error_count      2   0.00       0.000    0.0000     0.00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Allowed components list (restrict to these)
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# --- Helper to compute global p95 thresholds for a given dataframe ---
def compute_global_p95(df, comp_col, kpi_col):
    # ensure timestamp -> timezone-aware column if present
    if 'timestamp' in df.columns:
        try:
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        except Exception:
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    # filter components
    df = df[df[comp_col].isin(allowed_components)].copy()
    # ensure value column exists
    if 'value' not in df.columns:
        raise ValueError("Missing 'value' column")
    # group and compute count and p95 across the entire series
    grp = df.groupby([comp_col, kpi_col])['value']
    agg = grp.agg(
        count='count',
        p95_value=lambda x: np.percentile(x.dropna(), 95) if x.dropna().size>0 else np.nan
    ).reset_index()
    agg = agg.rename(columns={comp_col: 'component', kpi_col: 'kpi_name'})
    # round p95 for compactness
    agg['p95_value'] = agg['p95_value'].round(6)
    return agg

# --- Load files and compute thresholds ---
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')

metric_thresholds = compute_global_p95(metric_df, 'cmdb_id', 'kpi_name')
trace_thresholds  = compute_global_p95(trace_df,  'cmdb_id', 'trace_name')
log_thresholds    = compute_global_p95(log_df,    'cmdb_id', 'log_name')

# Prepare compact top-20 tables sorted by p95_value descending
metric_top = metric_thresholds.sort_values('p95_value', ascending=False).head(20)[['component','kpi_name','p95_value','count']]
trace_top  = trace_thresholds.sort_values('p95_value', ascending=False).head(20)[['component','kpi_name','p95_value','count']]
log_top    = log_thresholds.sort_values('p95_value', ascending=False).head(20)[['component','kpi_name','p95_value','count']]

# Display compact tables (up to 20 rows each)
metric_top, trace_top, log_top

```
Out[2]:
```
```python
# Compact plain-English summary of the computed global P95 thresholds
# Uses existing kernel variables: metric_thresholds, trace_thresholds, log_thresholds

def top_lines(df, n=5):
    rows = []
    for _, r in df.sort_values('p95_value', ascending=False).head(n).iterrows():
        pv = r['p95_value']
        # format p95 compactly
        if abs(pv) >= 1e6 or (pv != 0 and abs(pv) < 1e-3):
            pv_s = f"{pv:.6e}"
        else:
            pv_s = f"{pv:.6g}"
        rows.append(f"{r['component']} | {r['kpi_name']} | p95={pv_s} | count={int(r['count'])}")
    return "\n  ".join(rows) if rows else "  (none)"

summary = (
    "Global P95 thresholds (full series) — top candidates per file:\n\n"
    "Metric CSV — top by p95_value:\n"
    f"  {top_lines(metric_thresholds, n=5)}\n\n"
    "Trace CSV — top by p95_value:\n"
    f"  {top_lines(trace_thresholds, n=5)}\n\n"
    "Log CSV — top by p95_value:\n"
    f"  {top_lines(log_thresholds, n=5)}\n\n"
    "Notes:\n"
    "- Threshold tables are stored in kernel variables: metric_thresholds, trace_thresholds, log_thresholds.\n"
    "- Listings above show component, KPI name, computed global p95, and sample count (full available data).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     component                                           kpi_name     p95_value  count
878    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  5.814783e+09     16
879    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  5.279638e+09     16
913    Redis01  redis-Redis_6379_Redis  (total_commands_proces...  5.141719e+09     30
1017   Redis02                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     30
906    Redis01                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     30
1338  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1123  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1227  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1229  Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1449  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1121  Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1340  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1447  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1024   Redis02  redis-Redis_6379_Redis  (total_commands_proces...  4.070112e+09     30
901    Redis01            redis-Redis_6379_Redis  (keyspace_hits)  3.910685e+09     30
1337  Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  3.148860e+09      4
1446  Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  3.019656e+09      5
1012   Redis02            redis-Redis_6379_Redis  (keyspace_hits)  2.944588e+09     30
1226  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  2.844460e+09      5
1120  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  2.676308e+09      4,     component                       kpi_name  p95_value  count
131  Tomcat01           trace.self.row_count    3508.45     30
185  Tomcat04           trace.self.row_count    3478.20     30
149  Tomcat02           trace.self.row_count    3388.00     30
167  Tomcat03           trace.self.row_count    3354.55     30
5        IG01           trace.self.row_count    2018.10     30
23       IG02           trace.self.row_count    1941.60     30
65       MG01           trace.self.row_count    1413.85     30
107      MG02           trace.self.row_count    1343.60     30
11       IG01    trace.to_Tomcat02.row_count     261.65     30
140  Tomcat02      trace.from_IG01.row_count     261.65     30
89       MG02  trace.from_Tomcat04.row_count     259.85     30
47       MG01  trace.from_Tomcat04.row_count     259.85     30
191  Tomcat04        trace.to_MG02.row_count     259.85     30
188  Tomcat04        trace.to_MG01.row_count     259.85     30
35       IG02    trace.to_Tomcat04.row_count     258.85     30
179  Tomcat04      trace.from_IG02.row_count     258.85     30
134  Tomcat01        trace.to_MG01.row_count     257.75     30
38       MG01  trace.from_Tomcat01.row_count     257.75     30
8        IG01    trace.to_Tomcat01.row_count     257.50     30
122  Tomcat01      trace.from_IG01.row_count     257.50     30,    component          kpi_name  p95_value  count
17  apache01     log.row_count   993.5000     31
19  apache02     log.row_count   983.0000     31
6   Tomcat01     log.row_count   523.0000     31
15  Tomcat04     log.row_count   515.0000     31
12  Tomcat03     log.row_count   493.5000     31
9   Tomcat02     log.row_count   492.5000     31
1    Mysql01     log.row_count    14.8000     25
3    Mysql02     log.row_count     1.0000      2
5   Tomcat01  log.gc_pause_max     0.1180     23
8   Tomcat02  log.gc_pause_max     0.0955     30
14  Tomcat04  log.gc_pause_max     0.0900     22
11  Tomcat03  log.gc_pause_max     0.0890     23
2    Mysql02   log.error_count     0.0000      2
0    Mysql01   log.error_count     0.0000     25
4   Tomcat01   log.error_count     0.0000     31
7   Tomcat02   log.error_count     0.0000     31
10  Tomcat03   log.error_count     0.0000     31
13  Tomcat04   log.error_count     0.0000     31
16  apache01   log.error_count     0.0000     31
18  apache02   log.error_count     0.0000     31)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Incident window in UTC+8 (inclusive)
start_local = tz.localize(pd.Timestamp("2021-03-10 14:00:00"))
end_local   = tz.localize(pd.Timestamp("2021-03-10 14:30:00"))

# Helper: load and prepare a telemetry dataframe (adds tz-aware 'ts')
def load_prepare(filename):
    df = pd.read_csv(filename)
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

# Load sources
metric_df = load_prepare('metric.csv')
trace_df  = load_prepare('trace.csv')
log_df    = load_prepare('log.csv')
error_df  = load_prepare('error_logs.csv')

# Filter by allowed components and time window (inclusive) and save requested kernel variables
def filter_window(df, comp_col='cmdb_id'):
    d = df[df[comp_col].isin(allowed_components)].copy()
    d = d[(d['ts'] >= start_local) & (d['ts'] <= end_local)]
    return d

window_metric = filter_window(metric_df, 'cmdb_id')
window_trace  = filter_window(trace_df,  'cmdb_id')
window_log    = filter_window(log_df,    'cmdb_id')
window_error_logs = filter_window(error_df, 'cmdb_id')

# Ensure thresholds exist in kernel; if not, compute them from full series (no time filter)
def compute_global_p95(df, comp_col, kpi_col):
    dfp = df[df[comp_col].isin(allowed_components)].copy()
    grp = dfp.groupby([comp_col, kpi_col])['value']
    agg = grp.agg(count='count', p95_value=lambda x: np.percentile(x.dropna(),95) if x.dropna().size>0 else np.nan).reset_index()
    agg = agg.rename(columns={comp_col:'component', kpi_col:'kpi_name'})
    agg['p95_value'] = agg['p95_value'].round(6)
    return agg

try:
    metric_thresholds
except NameError:
    metric_thresholds = compute_global_p95(metric_df, 'cmdb_id', 'kpi_name')
    trace_thresholds  = compute_global_p95(trace_df,  'cmdb_id', 'trace_name')
    log_thresholds    = compute_global_p95(log_df,    'cmdb_id', 'log_name')

# Prepare window tables with unified column names for joining
w_metric = window_metric.rename(columns={'cmdb_id':'component'})[['component','kpi_name','value','timestamp','ts']].copy()
w_trace  = window_trace.rename(columns={'cmdb_id':'component','trace_name':'kpi_name'})[['component','kpi_name','value','timestamp','ts']].copy()
w_log    = window_log.rename(columns={'cmdb_id':'component','log_name':'kpi_name'})[['component','kpi_name','value','timestamp','ts']].copy()

# Generic function to detect anomalies by joining with thresholds
def detect_anomalies(window_df, thresholds_df):
    merged = pd.merge(window_df, thresholds_df[['component','kpi_name','p95_value']], on=['component','kpi_name'], how='left')
    merged['value_num'] = pd.to_numeric(merged['value'], errors='coerce')
    merged['is_anomaly'] = False
    valid_mask = merged['p95_value'].notna() & merged['value_num'].notna()
    merged.loc[valid_mask, 'is_anomaly'] = merged.loc[valid_mask, 'value_num'] >= merged.loc[valid_mask, 'p95_value']
    # group and summarize via apply (returns compact summary)
    def summarize_group(g):
        count_points = int(g['value_num'].count())
        count_anom = int(g['is_anomaly'].sum())
        earliest = g.loc[g['is_anomaly'], 'ts'].min() if count_anom>0 else pd.NaT
        max_val = g['value_num'].max() if g['value_num'].notna().any() else np.nan
        p95 = g['p95_value'].iloc[0] if not g['p95_value'].empty else np.nan
        return pd.Series({
            'count_points_in_window': count_points,
            'count_anomalies': count_anom,
            'earliest_anomaly_ts': earliest,
            'max_value_in_window': max_val,
            'p95_value': p95
        })
    summary = merged.groupby(['component','kpi_name']).apply(summarize_group).reset_index()
    # format earliest timestamp to tz-aware or NaT
    def fmt_ts(x):
        if pd.isna(x) or x is pd.NaT:
            return pd.NaT
        return pd.to_datetime(x).tz_convert(tz)
    summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_ts'].apply(fmt_ts)
    summary = summary.drop(columns=['earliest_anomaly_ts'])
    # compute ratio
    def ratio(row):
        pv = row['p95_value']
        mv = row['max_value_in_window']
        if pd.isna(mv):
            return np.nan
        if pd.isna(pv):
            return np.inf
        try:
            if pv == 0:
                return np.inf
            return mv / pv
        except Exception:
            return np.inf
    summary['max_value_over_p95'] = summary.apply(ratio, axis=1)
    # round numeric cols for compactness
    for c in ['max_value_in_window','p95_value','max_value_over_p95']:
        summary[c] = pd.to_numeric(summary[c], errors='coerce').round(6)
    # sort by count_anomalies desc then by ratio desc
    summary = summary.sort_values(['count_anomalies','max_value_over_p95'], ascending=[False,False])
    return summary

metric_anom_summary = detect_anomalies(w_metric, metric_thresholds)
trace_anom_summary  = detect_anomalies(w_trace,  trace_thresholds)
log_anom_summary    = detect_anomalies(w_log,    log_thresholds)

# Compact top-20 tables with requested columns and ordering
metric_summary_top = metric_anom_summary[['component','kpi_name','count_points_in_window','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','p95_value','max_value_over_p95']].head(20)
trace_summary_top  = trace_anom_summary[['component','kpi_name','count_points_in_window','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','p95_value','max_value_over_p95']].head(20)
log_summary_top    = log_anom_summary[['component','kpi_name','count_points_in_window','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','p95_value','max_value_over_p95']].head(20)

# Prepare first 20 rows of window_error_logs for context (component, timestamp, message)
window_error_logs = window_error_logs.rename(columns={'cmdb_id':'component'})
error_logs_head20 = window_error_logs[['component','ts','message']].head(20).rename(columns={'ts':'timestamp'})

# Display compact summaries (variables to be shown by the IPython kernel)
metric_summary_top, trace_summary_top, log_summary_top, error_logs_head20

```
Out[3]:
```
Summary of findings for incident window 2021-03-10 14:00 — 14:30 (UTC+8), restricted to allowed components:

1) metric.csv (window anomalies)
- Many metric series were flagged because their historical p95_value is 0, so all points in the window (31 samples) satisfy value >= p95. Examples: IG01 and IG02 TCP CLOSE‑WAIT and FIN‑WAIT, Mysql01/Mysql02 same TCP metrics, many Tomcat session counters. Most of those have max_value_in_window = 0.0 (so the alerts are numerically degenerate).
- Notable real non‑zero metric in the top list: Redis01 OSLinux NETWORK FIN‑WAIT had max=11.0 while p95=0 → flagged (ratio = inf). Interpretation: many metric anomalies here are artifacts of p95==0; only a few series show non‑zero maxima.

2) trace.csv (window anomalies)
- Clear signal in traces: several MG01 / MG02 and Tomcat trace duration or row_count KPIs exceeded their global p95 by large margins (ratios ~1.9–2.6).
- Examples (top candidates):
  - MG01 trace.from_dockerA2.row_count: 30 points, 3 anomalies, earliest anomaly 2021-03-10 14:15, max 62, p95 57, ratio ≈ 1.09.
  - MG02 trace.from_dockerA2.row_count: 30 points, 3 anomalies, earliest 14:02, max 62, p95 58, ratio ≈ 1.07.
  - Multiple duration_p95 KPIs (to/from docker and services) in MG01/MG02 and Tomcat nodes show 2x–2.6x above p95 with earliest anomalies typically ~14:09 or ~14:19.
- Interpretation: traces show real latency/load spikes concentrated around ~14:08–14:19 impacting MG nodes and some Tomcat endpoints.

3) log.csv (window anomalies)
- log.error_count series for many Tomcat/apache/mysql components are flagged because p95_value=0 and window values = 0 (i.e., all points >= p95). These are not meaningful as errors (zeros).
- Meaningful small anomalies:
  - GC pause maxima: Tomcat04 (p95=0.09, max=0.13, ratio≈1.44, earliest 14:08), Tomcat03 and Tomcat01 also show modest GC pause exceedances (~1.1–1.35x).
  - log.row_count for apache and Tomcat show small spikes above p95 (ratios ≈1.03–1.06) around ~14:12.
- Interpretation: logs show modest increases in request volumes and GC pause spikes; error_count zeros are instrumentation artifacts due to p95==0.

4) error_logs (context)
- No error_logs rows were found in the filtered window (the first 20 rows selection is empty).

Overall conclusion (concise):
- The strongest actionable signals come from trace metrics: MG01/MG02 and several Tomcat trace duration/row_count KPIs exceeded their global P95s by substantial factors in the 14:08–14:20 timeframe — these are likely real performance degradations to investigate first.
- Many metric and log “anomalies” are caused by p95_value == 0 (leading to all points flagged, often with values 0). Treat those with caution; they are typically not meaningful alerts.
- No error log entries were captured in the window.

If you want, I can:
- show the top 20 rows of each compact summary table again,
- filter trace anomalies to only those with count_anomalies >= 2 and ratio > 1.5 for prioritized investigation,
- or provide the raw matching timepoints for the top trace anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(     component                                           kpi_name  count_points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  p95_value  max_value_over_p95
66        IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
67        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
154       IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
155       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
572    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
573    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
796    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
797    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
799    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
869    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
870    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31               31  2021-03-10 14:00:00+08:00                 11.0        0.0                 inf
986    Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
987    Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
989    Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
1129  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
1130  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
1131  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
1132  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
1133  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf
1134  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions                      31               31  2021-03-10 14:00:00+08:00                  0.0        0.0                 inf,     component                          kpi_name  count_points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  p95_value  max_value_over_p95
53       MG01     trace.from_dockerA2.row_count                      30                3  2021-03-10 14:15:00+08:00            62.000000  57.000000            1.087719
95       MG02     trace.from_dockerA2.row_count                      30                3  2021-03-10 14:02:00+08:00            62.000000  58.000000            1.068966
67       MG01    trace.to_dockerA1.duration_p95                      30                2  2021-03-10 14:09:00+08:00            26.842000  10.381672            2.585518
94       MG02  trace.from_dockerA2.duration_p95                      30                2  2021-03-10 14:09:00+08:00            35.024400  13.744920            2.548171
100      MG02  trace.from_dockerB2.duration_p95                      30                2  2021-03-10 14:19:00+08:00            34.315400  13.590060            2.525037
49       MG01  trace.from_dockerA1.duration_p95                      30                2  2021-03-10 14:09:00+08:00            36.931000  14.769875            2.500427
58       MG01  trace.from_dockerB2.duration_p95                      30                2  2021-03-10 14:19:00+08:00            34.067900  13.832530            2.462883
97       MG02  trace.from_dockerB1.duration_p95                      30                2  2021-03-10 14:09:00+08:00            35.153950  14.643005            2.400733
70       MG01    trace.to_dockerA2.duration_p95                      30                2  2021-03-10 14:09:00+08:00            25.097000  11.015427            2.278350
55       MG01  trace.from_dockerB1.duration_p95                      30                2  2021-03-10 14:19:00+08:00            33.440900  15.009000            2.228056
85       MG02  trace.from_Tomcat03.duration_p95                      30                2  2021-03-10 14:09:00+08:00            37.676200  17.191647            2.191541
172  Tomcat03        trace.to_MG02.duration_p95                      30                2  2021-03-10 14:09:00+08:00            37.676200  17.191647            2.191541
106      MG02           trace.self.duration_p95                      30                2  2021-03-10 14:09:00+08:00            34.576400  15.893452            2.175512
64       MG01           trace.self.duration_p95                      30                2  2021-03-10 14:09:00+08:00            34.544500  16.204950            2.131725
112      MG02    trace.to_dockerA2.duration_p95                      30                2  2021-03-10 14:09:00+08:00            22.634400  10.738107            2.107858
91       MG02  trace.from_dockerA1.duration_p95                      30                2  2021-03-10 14:19:00+08:00            33.394650  17.405435            1.918633
37       MG01  trace.from_Tomcat01.duration_p95                      30                2  2021-03-10 14:09:00+08:00            32.972500  17.574087            1.876200
133  Tomcat01        trace.to_MG01.duration_p95                      30                2  2021-03-10 14:09:00+08:00            32.972500  17.574087            1.876200
166  Tomcat03           trace.self.duration_p95                      30                2  2021-03-10 14:08:00+08:00             5.014400   2.754202            1.820636
111      MG02   trace.to_dockerA2.duration_mean                      30                2  2021-03-10 14:19:00+08:00             3.203643   1.791213            1.788533,    component          kpi_name  count_points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  p95_value  max_value_over_p95
4   Tomcat01   log.error_count                      31               31  2021-03-10 14:00:00+08:00                 0.00     0.0000                 inf
7   Tomcat02   log.error_count                      31               31  2021-03-10 14:00:00+08:00                 0.00     0.0000                 inf
10  Tomcat03   log.error_count                      31               31  2021-03-10 14:00:00+08:00                 0.00     0.0000                 inf
13  Tomcat04   log.error_count                      31               31  2021-03-10 14:00:00+08:00                 0.00     0.0000                 inf
16  apache01   log.error_count                      31               31  2021-03-10 14:00:00+08:00                 0.00     0.0000                 inf
18  apache02   log.error_count                      31               31  2021-03-10 14:00:00+08:00                 0.00     0.0000                 inf
0    Mysql01   log.error_count                      25               25  2021-03-10 14:01:00+08:00                 0.00     0.0000                 inf
14  Tomcat04  log.gc_pause_max                      22                3  2021-03-10 14:08:00+08:00                 0.13     0.0900            1.444444
2    Mysql02   log.error_count                       2                2  2021-03-10 14:13:00+08:00                 0.00     0.0000                 inf
11  Tomcat03  log.gc_pause_max                      23                2  2021-03-10 14:02:00+08:00                 0.12     0.0890            1.348315
1    Mysql01     log.row_count                      25                2  2021-03-10 14:16:00+08:00                17.00    14.8000            1.148649
5   Tomcat01  log.gc_pause_max                      23                2  2021-03-10 14:18:00+08:00                 0.13     0.1180            1.101695
19  apache02     log.row_count                      31                2  2021-03-10 14:12:00+08:00              1040.00   983.0000            1.057986
9   Tomcat02     log.row_count                      31                2  2021-03-10 14:12:00+08:00               521.00   492.5000            1.057868
8   Tomcat02  log.gc_pause_max                      30                2  2021-03-10 14:13:00+08:00                 0.10     0.0955            1.047120
17  apache01     log.row_count                      31                2  2021-03-10 14:12:00+08:00              1033.00   993.5000            1.039758
15  Tomcat04     log.row_count                      31                2  2021-03-10 14:12:00+08:00               533.00   515.0000            1.034951
12  Tomcat03     log.row_count                      31                2  2021-03-10 14:12:00+08:00               508.00   493.5000            1.029382
6   Tomcat01     log.row_count                      31                2  2021-03-10 14:12:00+08:00               537.00   523.0000            1.026769
3    Mysql02     log.row_count                       2                2  2021-03-10 14:13:00+08:00                 1.00     1.0000            1.000000, Empty DataFrame
Columns: [component, timestamp, message]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone and variables from kernel
tz = pytz.timezone('Asia/Shanghai')

# Expecting these variables to exist from prior steps:
# trace_anom_summary, window_trace, trace_thresholds
# If any are missing, raise a clear error.
for var in ['trace_anom_summary', 'window_trace', 'trace_thresholds']:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel. Please run previous steps.")

# 1) Select candidate trace-faults: count_anomalies >=2 and max_value_over_p95 > 1.8
candidates = trace_anom_summary[
    (trace_anom_summary['count_anomalies'] >= 2) &
    (trace_anom_summary['max_value_over_p95'] > 1.8)
].copy()

# Ensure sorting by severity (ratio) descending
candidates = candidates.sort_values('max_value_over_p95', ascending=False)

# Select requested columns and limit to top 20
prioritized_trace_faults = candidates[['component','kpi_name','count_points_in_window','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','p95_value','max_value_over_p95']].head(20).copy()

# 2) For top 3 candidates, retrieve anomaly timepoints from window_trace where value >= p95_value
top3 = prioritized_trace_faults.head(3).reset_index(drop=True)

# Prepare window_trace (should contain original trace_name column and timestamp/ts/value)
# window_trace is the filtered full table with columns including 'trace_name', 'cmdb_id', 'value', 'timestamp', 'ts'
wt = window_trace.copy()

# Ensure numeric conversion
wt['value_num'] = pd.to_numeric(wt['value'], errors='coerce')

# Merge with trace_thresholds to get p95_value per series
wt_merge = pd.merge(
    wt,
    trace_thresholds[['component','kpi_name','p95_value']],
    left_on=['cmdb_id','trace_name'],
    right_on=['component','kpi_name'],
    how='left'
)

# Collect up to 10 anomaly points per top candidate
points_list = []
for idx, row in top3.iterrows():
    comp = row['component']
    kpi = row['kpi_name']
    # filter matching series
    mask = (wt_merge['cmdb_id'] == comp) & (wt_merge['trace_name'] == kpi)
    sub = wt_merge[mask].copy()
    # Ensure numeric and p95 present
    sub['p95_value'] = pd.to_numeric(sub['p95_value'], errors='coerce')
    # Keep only points where value >= p95_value and p95 is not NaN
    valid = sub[(sub['p95_value'].notna()) & (sub['value_num'].notna()) & (sub['value_num'] >= sub['p95_value'])]
    # sort by timestamp ascending and limit to 10
    valid = valid.sort_values('ts').head(10)
    if not valid.empty:
        # select requested columns: component, trace_name, timestamp, value
        out = valid[['cmdb_id','trace_name','ts','value_num']].copy()
        out = out.rename(columns={'cmdb_id':'component','ts':'timestamp','value_num':'value'})
        # attach candidate rank for traceability
        out['candidate_rank'] = idx+1
        out['kpi_name'] = kpi
        points_list.append(out)

if points_list:
    prioritized_trace_anomaly_points = pd.concat(points_list, ignore_index=True)
    # keep requested columns ordering: component, trace_name, timestamp, value (plus candidate_rank,kpi_name)
    prioritized_trace_anomaly_points = prioritized_trace_anomaly_points[['candidate_rank','component','kpi_name','trace_name','timestamp','value']].copy()
else:
    # empty DataFrame with requested columns
    prioritized_trace_anomaly_points = pd.DataFrame(columns=['candidate_rank','component','kpi_name','trace_name','timestamp','value'])

# Keep outputs compact for display
prioritized_trace_faults, prioritized_trace_anomaly_points.head(50)

```
Out[4]:
```
Focused trace-fault candidates (criteria: count_anomalies >= 2 and max_value_over_p95 > 1.8). Columns: component | kpi_name | count_points_in_window | count_anomalies | earliest_anomaly_timestamp | max_value_in_window | p95_value | max_value_over_p95

- MG01 | trace.to_dockerA1.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 26.84200 | 10.381672 | 2.585518
- MG02 | trace.from_dockerA2.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 35.02440 | 13.744920 | 2.548171
- MG02 | trace.from_dockerB2.duration_p95 | 30 | 2 | 2021-03-10 14:19:00+08:00 | 34.31540 | 13.590060 | 2.525037
- MG01 | trace.from_dockerA1.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 36.93100 | 14.769875 | 2.500427
- MG01 | trace.from_dockerB2.duration_p95 | 30 | 2 | 2021-03-10 14:19:00+08:00 | 34.06790 | 13.832530 | 2.462883
- MG02 | trace.from_dockerB1.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 35.15395 | 14.643005 | 2.400733
- MG01 | trace.to_dockerA2.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 25.09700 | 11.015427 | 2.278350
- MG01 | trace.from_dockerB1.duration_p95 | 30 | 2 | 2021-03-10 14:19:00+08:00 | 33.44090 | 15.009000 | 2.228056
- MG02 | trace.from_Tomcat03.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 37.67620 | 17.191647 | 2.191541
- Tomcat03 | trace.to_MG02.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 37.67620 | 17.191647 | 2.191541
- MG02 | trace.self.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 34.57640 | 15.893452 | 2.175512
- MG01 | trace.self.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 34.54450 | 16.204950 | 2.131725
- MG02 | trace.to_dockerA2.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 22.63440 | 10.738107 | 2.107858
- MG02 | trace.from_dockerA1.duration_p95 | 30 | 2 | 2021-03-10 14:19:00+08:00 | 33.39465 | 17.405435 | 1.918633
- MG01 | trace.from_Tomcat01.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 32.97250 | 17.574087 | 1.876200
- Tomcat01 | trace.to_MG01.duration_p95 | 30 | 2 | 2021-03-10 14:09:00+08:00 | 32.97250 | 17.574087 | 1.876200
- Tomcat03 | trace.self.duration_p95 | 30 | 2 | 2021-03-10 14:08:00+08:00 | 5.01440 | 2.754202 | 1.820636

Top 3 candidates (highest max_value_over_p95) and their anomaly timepoints (from window_trace; values shown, timestamps in UTC+8):

1) Rank 1 — MG01 | trace.to_dockerA1.duration_p95 (p95=10.381672, max=26.84200, ratio≈2.586)
   - 2021-03-10 14:09:00+08:00 — value 11.02515
   - 2021-03-10 14:19:00+08:00 — value 26.84200

2) Rank 2 — MG02 | trace.from_dockerA2.duration_p95 (p95=13.744920, max=35.02440, ratio≈2.548)
   - 2021-03-10 14:09:00+08:00 — value 14.18115
   - 2021-03-10 14:19:00+08:00 — value 35.02440

3) Rank 3 — MG02 | trace.from_dockerB2.duration_p95 (p95=13.590060, max=34.31540, ratio≈2.525)
   - 2021-03-10 14:19:00+08:00 — value 34.31540
   - 2021-03-10 14:26:00+08:00 — value 14.57700

Interpretation / next steps:
- These candidates point to latency spikes in MG01 and MG02 traces involving dockerA1/A2 and dockerB2 around ~14:09 and ~14:19–14:26. The spikes are ~1.9–2.6× above historical p95.
- Prioritize investigating MG routing, containers dockerA1/A2/B2, and related network or host resource issues in that 14:08–14:26 window. The kernel variables prioritized_trace_faults and prioritized_trace_anomaly_points contain the selected rows and the anomaly points for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(    component                          kpi_name  count_points_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  p95_value  max_value_over_p95
67       MG01    trace.to_dockerA1.duration_p95                      30                2  2021-03-10 14:09:00+08:00             26.84200  10.381672            2.585518
94       MG02  trace.from_dockerA2.duration_p95                      30                2  2021-03-10 14:09:00+08:00             35.02440  13.744920            2.548171
100      MG02  trace.from_dockerB2.duration_p95                      30                2  2021-03-10 14:19:00+08:00             34.31540  13.590060            2.525037
49       MG01  trace.from_dockerA1.duration_p95                      30                2  2021-03-10 14:09:00+08:00             36.93100  14.769875            2.500427
58       MG01  trace.from_dockerB2.duration_p95                      30                2  2021-03-10 14:19:00+08:00             34.06790  13.832530            2.462883
97       MG02  trace.from_dockerB1.duration_p95                      30                2  2021-03-10 14:09:00+08:00             35.15395  14.643005            2.400733
70       MG01    trace.to_dockerA2.duration_p95                      30                2  2021-03-10 14:09:00+08:00             25.09700  11.015427            2.278350
55       MG01  trace.from_dockerB1.duration_p95                      30                2  2021-03-10 14:19:00+08:00             33.44090  15.009000            2.228056
85       MG02  trace.from_Tomcat03.duration_p95                      30                2  2021-03-10 14:09:00+08:00             37.67620  17.191647            2.191541
172  Tomcat03        trace.to_MG02.duration_p95                      30                2  2021-03-10 14:09:00+08:00             37.67620  17.191647            2.191541
106      MG02           trace.self.duration_p95                      30                2  2021-03-10 14:09:00+08:00             34.57640  15.893452            2.175512
64       MG01           trace.self.duration_p95                      30                2  2021-03-10 14:09:00+08:00             34.54450  16.204950            2.131725
112      MG02    trace.to_dockerA2.duration_p95                      30                2  2021-03-10 14:09:00+08:00             22.63440  10.738107            2.107858
91       MG02  trace.from_dockerA1.duration_p95                      30                2  2021-03-10 14:19:00+08:00             33.39465  17.405435            1.918633
37       MG01  trace.from_Tomcat01.duration_p95                      30                2  2021-03-10 14:09:00+08:00             32.97250  17.574087            1.876200
133  Tomcat01        trace.to_MG01.duration_p95                      30                2  2021-03-10 14:09:00+08:00             32.97250  17.574087            1.876200
166  Tomcat03           trace.self.duration_p95                      30                2  2021-03-10 14:08:00+08:00              5.01440   2.754202            1.820636,    candidate_rank component                          kpi_name                        trace_name                 timestamp     value
0               1      MG01    trace.to_dockerA1.duration_p95    trace.to_dockerA1.duration_p95 2021-03-10 14:09:00+08:00  11.02515
1               1      MG01    trace.to_dockerA1.duration_p95    trace.to_dockerA1.duration_p95 2021-03-10 14:19:00+08:00  26.84200
2               2      MG02  trace.from_dockerA2.duration_p95  trace.from_dockerA2.duration_p95 2021-03-10 14:09:00+08:00  14.18115
3               2      MG02  trace.from_dockerA2.duration_p95  trace.from_dockerA2.duration_p95 2021-03-10 14:19:00+08:00  35.02440
4               3      MG02  trace.from_dockerB2.duration_p95  trace.from_dockerB2.duration_p95 2021-03-10 14:19:00+08:00  34.31540
5               3      MG02  trace.from_dockerB2.duration_p95  trace.from_dockerB2.duration_p95 2021-03-10 14:26:00+08:00  14.57700)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timezone-aware timestamp per rule 10 (UTC+8) though we won't filter by time
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Components of interest
components_of_interest = ["MG01","MG02","Tomcat01","Tomcat03","Tomcat04"]

# Filter to these components (no time filtering as requested)
sel = metric_df[metric_df['cmdb_id'].isin(components_of_interest)].copy()

# Aggregate by (cmdb_id, kpi_name) using full available data
grp = sel.groupby(['cmdb_id','kpi_name'])['value']

selected_metric_groups = grp.agg(
    count='count',
    min_value=lambda x: np.nanmin(x.astype(float)) if x.dropna().size>0 else np.nan,
    median_p50=lambda x: np.percentile(x.dropna().astype(float), 50) if x.dropna().size>0 else np.nan,
    p95=lambda x: np.percentile(x.dropna().astype(float), 95) if x.dropna().size>0 else np.nan,
    max_value=lambda x: np.nanmax(x.astype(float)) if x.dropna().size>0 else np.nan
).reset_index().rename(columns={
    'cmdb_id':'component',
    'kpi_name':'kpi_name',
    'min_value':'min',
    'max_value':'max'
})

# Round numeric columns for compactness
for c in ['min','median_p50','p95','max']:
    if c in selected_metric_groups.columns:
        selected_metric_groups[c] = pd.to_numeric(selected_metric_groups[c], errors='coerce').round(6)

# Sort by component then p95 descending
selected_metric_groups = selected_metric_groups.sort_values(['component','p95'], ascending=[True, False])

# Keep up to 50 rows for display as requested
selected_metric_groups.head(50)

```
Out[5]:
```
```python
# Summarize available metric KPI groups for MG01, MG02, Tomcat01, Tomcat03, Tomcat04
# Uses the kernel variable `selected_metric_groups` produced earlier.
import math

components = ["MG01","MG02","Tomcat01","Tomcat03","Tomcat04"]

if 'selected_metric_groups' not in globals():
    raise NameError("Kernel variable 'selected_metric_groups' not found. Please run the previous step to create it.")

df = selected_metric_groups.copy()

def fmt_num(x):
    try:
        if pd.isna(x):
            return "nan"
        x = float(x)
        if x == 0:
            return "0"
        if abs(x) >= 1e6 or abs(x) < 1e-3:
            return f"{x:.6e}"
        return f"{x:.6g}"
    except Exception:
        return str(x)

lines = []
total_rows = len(df)
lines.append(f"Metric KPI groups (full data) — components: {', '.join(components)}")
lines.append(f"Total KPI groups found: {total_rows}")
lines.append("")

# count KPI groups per component
counts = df['component'].value_counts().reindex(components).fillna(0).astype(int)
lines.append("KPI group counts per component:")
for c in components:
    lines.append(f"- {c}: {counts[c]}")

lines.append("")
lines.append("Top KPI groups by p95 per component (up to 3 each):")
for c in components:
    sub = df[df['component']==c].sort_values('p95', ascending=False).head(3)
    if sub.empty:
        lines.append(f"- {c}: (no KPI groups)")
        continue
    lines.append(f"- {c}:")
    for _, r in sub.iterrows():
        lines.append(
            f"  • {r['kpi_name']} | count={int(r['count'])} | min={fmt_num(r['min'])} | p50={fmt_num(r['median_p50'])} | p95={fmt_num(r['p95'])} | max={fmt_num(r['max'])}"
        )

lines.append("")
lines.append("Note: Full aggregated table is stored in kernel variable `selected_metric_groups` (use for detailed inspection).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

component                                           kpi_name  count           min    median_p50           p95           max
0       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
2       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed     31  2.448990e+08  6.309664e+08  9.860176e+08  1.063591e+09
3       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed     31  5.069445e+08  5.069445e+08  5.069445e+08  5.069445e+08
93      MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     20  6.555733e+06  6.556606e+06  6.557422e+06  6.557479e+06
5       MG01                    JVM-Runtime_7779_JVM_JVM_Uptime     20  2.419058e+06  2.419929e+06  2.420682e+06  2.420738e+06
..       ...                                                ...    ...           ...           ...           ...           ...
64      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite      9  9.600000e+00  9.600000e+00  9.600000e+00  9.600000e+00
61      MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...     11  9.000000e+00  9.000000e+00  9.500000e+00  1.000000e+01
9       MG01                    OSLinux-CPU_CPU-1_SingleCpuUtil     11  3.393800e+00  4.761100e+00  6.632400e+00  7.117900e+00
7       MG01                    OSLinux-CPU_CPU-0_SingleCpuUtil     10  1.597100e+00  3.368800e+00  5.635715e+00  6.257300e+00
57      MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps     10  0.000000e+00  0.000000e+00  4.000000e+00  4.000000e+00

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse timezone
tz = pytz.timezone('Asia/Shanghai')

# Components of interest
components_of_interest = ["MG01","MG02","Tomcat01","Tomcat03","Tomcat04"]

# Ensure required kernel variables exist
if 'window_metric' not in globals():
    # load and filter as fallback (incident window Mar 10 14:00-14:30 UTC+8)
    metric_df = pd.read_csv('metric.csv')
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_local = tz.localize(pd.Timestamp("2021-03-10 14:00:00"))
    end_local   = tz.localize(pd.Timestamp("2021-03-10 14:30:00"))
    window_metric = metric_df[(metric_df['ts'] >= start_local) & (metric_df['ts'] <= end_local)].copy()

if 'metric_thresholds' not in globals():
    full_metric_df = pd.read_csv('metric.csv')
    grp = full_metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_thresholds = grp.agg(count='count', p95_value=lambda x: np.percentile(x.dropna().astype(float),95) if x.dropna().size>0 else np.nan).reset_index()
    metric_thresholds = metric_thresholds.rename(columns={'cmdb_id':'component'})

# Prepare window data for selected components
wm = window_metric.copy()
if 'ts' not in wm.columns:
    wm['ts'] = pd.to_datetime(wm['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    wm['ts'] = pd.to_datetime(wm['ts']).dt.tz_convert(tz)

wm = wm[wm['cmdb_id'].isin(components_of_interest)].copy()
wm['value_num'] = pd.to_numeric(wm['value'], errors='coerce')

# Merge with metric_thresholds to get global p95 per series
mt = metric_thresholds[['component','kpi_name','p95_value']].copy()
merged = pd.merge(wm, mt, left_on=['cmdb_id','kpi_name'], right_on=['component','kpi_name'], how='left')

# Mark anomalies where value >= p95_value (require p95_value not NaN)
merged['is_anom'] = False
valid_mask = merged['p95_value'].notna() & merged['value_num'].notna()
merged.loc[valid_mask, 'is_anom'] = merged.loc[valid_mask, 'value_num'] >= merged.loc[valid_mask, 'p95_value']

# Identify consecutive anomalous runs per (cmdb_id, kpi_name)
runs = []
group_cols = ['cmdb_id','kpi_name']
for (comp, kpi), g in merged.groupby(group_cols):
    g = g.sort_values('ts').reset_index(drop=True)
    if g.empty:
        continue
    # segment id increments when is_anom changes
    change = (g['is_anom'] != g['is_anom'].shift(fill_value=False))
    seg_id = change.cumsum()
    g['seg_id'] = seg_id
    # iterate anomalous segments
    anom_groups = g[g['is_anom']].groupby('seg_id')
    for seg, seg_df in anom_groups:
        run_length = int(seg_df.shape[0])
        if run_length < 2:
            continue  # keep only runs with length >= 2
        run_start = seg_df['ts'].iloc[0]
        run_end = seg_df['ts'].iloc[-1]
        max_val = seg_df['value_num'].max()
        p95 = seg_df['p95_value'].iloc[0]
        # compute ratio, treating p95==0 as inf, NaN preserved
        if pd.isna(p95):
            ratio = np.nan
        else:
            try:
                ratio = np.inf if p95 == 0 else float(max_val) / float(p95)
            except Exception:
                ratio = np.inf
        runs.append({
            'component': comp,
            'kpi_name': kpi,
            'run_start_timestamp': run_start,
            'run_end_timestamp': run_end,
            'run_length': run_length,
            'max_value_in_run': max_val,
            'p95_value': p95,
            'max_over_p95': ratio
        })

# Store full results in kernel variable metric_faults
metric_faults = pd.DataFrame(runs) if runs else pd.DataFrame(columns=[
    'component','kpi_name','run_start_timestamp','run_end_timestamp',
    'run_length','max_value_in_run','p95_value','max_over_p95'
])

# Filter to significant breaches: max_over_p95 > 1.5 (inf counts as >1.5)
# Create a numeric comparison that treats inf appropriately
def is_significant(x):
    if pd.isna(x):
        return False
    try:
        if np.isinf(x):
            return True
        return float(x) > 1.5
    except Exception:
        return False

if not metric_faults.empty:
    metric_faults['is_significant'] = metric_faults['max_over_p95'].apply(is_significant)
    significant = metric_faults[metric_faults['is_significant']].copy()
else:
    significant = metric_faults.copy()

# Prepare compact sorted table
if not significant.empty:
    # Normalize numeric columns for display
    for c in ['max_value_in_run','p95_value','max_over_p95']:
        if c in significant.columns:
            significant[c] = pd.to_numeric(significant[c], errors='coerce')
    # Sort by max_over_p95 desc (inf first) then run_length desc
    significant = significant.sort_values(by=['max_over_p95','run_length'], ascending=[False, False])
    # Format timestamps as tz-aware
    significant['run_start_timestamp'] = pd.to_datetime(significant['run_start_timestamp']).dt.tz_convert(tz)
    significant['run_end_timestamp'] = pd.to_datetime(significant['run_end_timestamp']).dt.tz_convert(tz)
    # Select columns and round numeric values for compactness
    metric_faults_top = significant[['component','kpi_name','run_start_timestamp','run_end_timestamp','run_length','max_value_in_run','p95_value','max_over_p95']].head(20).copy()
    metric_faults_top['max_value_in_run'] = metric_faults_top['max_value_in_run'].round(6)
    metric_faults_top['p95_value'] = metric_faults_top['p95_value'].round(6)
    # keep max_over_p95 as is (inf preserved), but round finite values
    metric_faults_top['max_over_p95'] = metric_faults_top['max_over_p95'].apply(lambda x: np.inf if np.isinf(x) else (round(x,6) if not pd.isna(x) else x))
else:
    metric_faults_top = significant.head(20)

# Display compact result (metric_faults stored in kernel; metric_faults_top returned)
metric_faults_top

```
Out[6]:
```
Summary of metric-level fault runs in the incident window (components: MG01, MG02, Tomcat01, Tomcat03, Tomcat04):

- Detected fault runs are almost entirely session-related Tomcat metrics (Tomcat01 and Tomcat03) that are constant zero across the whole window (run_length = 31, run_start 14:00, run_end 14:30). Because the historical p95 for these KPIs is also 0, the computed ratio max_over_p95 is infinite — this is a degenerate artifact, not a meaningful breach.
- No meaningful metric fault runs with finite p95 breaches > 1.5 were found for the selected components in this window (i.e., no runs where max_value_in_run / p95_value > 1.5 with a finite p95_value).
- The full runs table is stored in kernel variable metric_faults. The compact top results were returned as metric_faults_top (showing the session KPIs with inf ratios).

Recommendation:
- Ignore series where p95_value == 0 (instrumentation/normal-zero counters) or treat them separately.
- If you want, I can re-run detection but (a) exclude series with p95==0, or (b) show only runs with finite p95 and max_over_p95 > 1.2 to surface subtler breaches. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

component                                           kpi_name       run_start_timestamp         run_end_timestamp  run_length  max_value_in_run  p95_value  max_over_p95
182  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
183  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
184  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
185  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
186  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
187  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
188  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
189  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
190  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
244  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
245  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
246  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
247  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
248  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
249  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
250  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
251  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
252  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
253  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf
254  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-10 14:00:00+08:00 2021-03-10 14:30:00+08:00          31               0.0        0.0           inf```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure window_trace and window_log exist; if not, load and filter incident window as fallback
# (Fallback uses same window: 2021-03-10 14:00 — 14:30 UTC+8)
start_local = tz.localize(pd.Timestamp("2021-03-10 14:00:00"))
end_local   = tz.localize(pd.Timestamp("2021-03-10 14:30:00"))

if 'window_trace' not in globals():
    # load full trace and filter
    tmp = pd.read_csv('trace.csv')
    tmp['ts'] = pd.to_datetime(tmp['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_trace = tmp[(tmp['ts'] >= start_local) & (tmp['ts'] <= end_local)].copy()

if 'window_log' not in globals():
    tmp = pd.read_csv('log.csv')
    tmp['ts'] = pd.to_datetime(tmp['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_log = tmp[(tmp['ts'] >= start_local) & (tmp['ts'] <= end_local)].copy()

# 1) From window_trace: Tomcat03 where trace_name contains 'trace.self'
t3_traces = window_trace.copy()
# normalize columns if necessary
if 'trace_name' not in t3_traces.columns and 'trace' in t3_traces.columns:
    t3_traces = t3_traces.rename(columns={'trace':'trace_name'})

mask1 = (t3_traces['cmdb_id'] == 'Tomcat03') & (t3_traces['trace_name'].astype(str).str.contains('trace.self', na=False))
tomcat3_self_traces_window = (
    t3_traces[mask1]
    .assign(ts = pd.to_datetime(t3_traces.loc[mask1,'ts']).dt.tz_convert(tz))
    .loc[mask1, ['cmdb_id','trace_name','ts','value']]
    .rename(columns={'cmdb_id':'component','ts':'timestamp'})
    .sort_values('timestamp', ascending=True)
)
# round value for compactness
tomcat3_self_traces_window['value'] = pd.to_numeric(tomcat3_self_traces_window['value'], errors='coerce').round(6)
tomcat3_self_traces_window = tomcat3_self_traces_window.head(20).reset_index(drop=True)

# 2) From window_log: Tomcat03 and Tomcat04 for specific log_names
logs = window_log.copy()
# normalize column names
if 'log_name' not in logs.columns and 'log' in logs.columns:
    logs = logs.rename(columns={'log':'log_name'})

wanted_components = ['Tomcat03','Tomcat04']
wanted_log_names = ['log.gc_pause_max','log.row_count','log.error_count']

mask2 = logs['cmdb_id'].isin(wanted_components) & logs['log_name'].isin(wanted_log_names)
tomcat3_4_logs_window = (
    logs[mask2]
    .assign(ts = pd.to_datetime(logs.loc[mask2,'ts']).dt.tz_convert(tz))
    .loc[mask2, ['cmdb_id','log_name','ts','value']]
    .rename(columns={'cmdb_id':'component','ts':'timestamp'})
    .sort_values('timestamp', ascending=True)
)
tomcat3_4_logs_window['value'] = pd.to_numeric(tomcat3_4_logs_window['value'], errors='coerce').round(6)
tomcat3_4_logs_window = tomcat3_4_logs_window.head(40).reset_index(drop=True)

# Return the two compact tables (stored in kernel variables as requested)
tomcat3_self_traces_window, tomcat3_4_logs_window

```
Out[7]:
```
```python
# Summarize selected timepoints to disambiguate order and likely cause (plain English string)
# Uses kernel variables tomcat3_self_traces_window and tomcat3_4_logs_window produced earlier.

import pandas as pd
import numpy as np

# Validate presence
for var in ['tomcat3_self_traces_window','tomcat3_4_logs_window']:
    if var not in globals():
        raise NameError(f"Required kernel variable '{var}' not found.")

tr = tomcat3_self_traces_window.copy()
lg = tomcat3_4_logs_window.copy()

# Helper to get max info per trace/log series
def max_info(df, value_col='value', time_col='timestamp'):
    if df.empty:
        return None
    idx = df[value_col].idxmax()
    return df.loc[idx, value_col], df.loc[idx, time_col]

# Summarize trace.self series for Tomcat03
trace_summary_lines = []
trace_names = tr['trace_name'].unique()
for name in trace_names:
    sub = tr[tr['trace_name'] == name].sort_values('timestamp')
    if sub.empty:
        continue
    cnt = len(sub)
    vmin = sub['value'].min()
    vmax = sub['value'].max()
    tmin = sub['timestamp'].min()
    tmax = sub['timestamp'].max()
    vmax_val, vmax_ts = max_info(sub, 'value', 'timestamp')
    trace_summary_lines.append(
        f"- {name}: {cnt} points from {tmin} to {tmax}; min={vmin:.6g}, max={vmax:.6g} at {vmax_ts}"
    )

# Summarize logs for Tomcat03 and Tomcat04 (selected log_names)
log_summary_lines = []
for (comp, lname), g in lg.groupby(['component','log_name']):
    if g.empty:
        continue
    cnt = len(g)
    vmin = g['value'].min()
    vmax = g['value'].max()
    tmin = g['timestamp'].min()
    tmax = g['timestamp'].max()
    vmax_val, vmax_ts = max_info(g, 'value', 'timestamp')
    log_summary_lines.append(
        f"- {comp} | {lname}: {cnt} points from {tmin} to {tmax}; min={vmin:.6g}, max={vmax:.6g} at {vmax_ts}"
    )

# Compose final plain-English summary
summary_lines = [
    "Tomcat03 self-trace observations (incident window, timestamps in UTC+8):",
]
summary_lines += trace_summary_lines
summary_lines += ["", "Tomcat03 / Tomcat04 log observations (selected metrics):"]
summary_lines += log_summary_lines
summary_lines += [
    "",
    "Interpretation (concise):",
    "- Tomcat03 trace.self shows transient latency increases: duration_p95 rises to ~1.0–1.22s (peak in the sampled window) with duration_mean remaining much lower (~0.05–0.24s).",
    "- The trace.self row_count for Tomcat03 fluctuates: high at ~3,000 at 14:00–14:01, dips to ~2,290–2,705 around 14:02–14:04, then partially recovers (example 2,851 at 14:05). This suggests request throughput variation coinciding with latency spikes (~14:02–14:06).",
    "- Logs (Tomcat03/04) show no error_count spikes (error_count = 0 in sampled points). GC pause maxima are small (~0.07–0.08s in the sampled rows) and log.row_count (log lines) are ~400–500 — modest and not indicative of large GC or error storms in this slice.",
    "",
    "Conclusion: For Tomcat03 in this window, there are short-lived latency spikes (duration_p95 up to ~1.2s) that align with drops in trace.row_count around 14:02–14:06. Logs do not show corresponding error spikes or large GC pauses in the same minute-level slices, so investigate upstream load / request routing or transient backend delays around 14:02–14:06 for Tomcat03."
]

summary = "\n".join(summary_lines)

# Output the summary string compactly
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   component                trace_name                 timestamp        value
0   Tomcat03  trace.self.duration_mean 2021-03-10 14:00:00+08:00     0.057156
1   Tomcat03   trace.self.duration_p95 2021-03-10 14:00:00+08:00     0.047000
2   Tomcat03      trace.self.row_count 2021-03-10 14:00:00+08:00  3043.000000
3   Tomcat03  trace.self.duration_mean 2021-03-10 14:01:00+08:00     0.061440
4   Tomcat03   trace.self.duration_p95 2021-03-10 14:01:00+08:00     0.048600
5   Tomcat03      trace.self.row_count 2021-03-10 14:01:00+08:00  3149.000000
6   Tomcat03  trace.self.duration_mean 2021-03-10 14:02:00+08:00     0.089175
7   Tomcat03   trace.self.duration_p95 2021-03-10 14:02:00+08:00     1.003800
8   Tomcat03      trace.self.row_count 2021-03-10 14:02:00+08:00  2705.000000
9   Tomcat03      trace.self.row_count 2021-03-10 14:03:00+08:00  2293.000000
10  Tomcat03   trace.self.duration_p95 2021-03-10 14:03:00+08:00     1.068400
11  Tomcat03  trace.self.duration_mean 2021-03-10 14:03:00+08:00     0.206165
12  Tomcat03  trace.self.duration_mean 2021-03-10 14:04:00+08:00     0.113478
13  Tomcat03   trace.self.duration_p95 2021-03-10 14:04:00+08:00     1.005000
14  Tomcat03      trace.self.row_count 2021-03-10 14:04:00+08:00  2290.000000
15  Tomcat03  trace.self.duration_mean 2021-03-10 14:05:00+08:00     0.203281
16  Tomcat03   trace.self.duration_p95 2021-03-10 14:05:00+08:00     1.058500
17  Tomcat03      trace.self.row_count 2021-03-10 14:05:00+08:00  2851.000000
18  Tomcat03  trace.self.duration_mean 2021-03-10 14:06:00+08:00     0.235773
19  Tomcat03   trace.self.duration_p95 2021-03-10 14:06:00+08:00     1.224900,    component          log_name                 timestamp   value
0   Tomcat03   log.error_count 2021-03-10 14:00:00+08:00    0.00
1   Tomcat03  log.gc_pause_max 2021-03-10 14:00:00+08:00    0.07
2   Tomcat03     log.row_count 2021-03-10 14:00:00+08:00  457.00
3   Tomcat04   log.error_count 2021-03-10 14:00:00+08:00    0.00
4   Tomcat04     log.row_count 2021-03-10 14:00:00+08:00  433.00
..       ...               ...                       ...     ...
35  Tomcat04   log.error_count 2021-03-10 14:06:00+08:00    0.00
36  Tomcat04  log.gc_pause_max 2021-03-10 14:06:00+08:00    0.08
37  Tomcat04     log.row_count 2021-03-10 14:06:00+08:00  430.00
38  Tomcat04     log.row_count 2021-03-10 14:07:00+08:00  424.00
39  Tomcat03     log.row_count 2021-03-10 14:07:00+08:00  428.00

[40 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure window_metric exists; if not, load and filter incident window as fallback
if 'window_metric' not in globals():
    df_full = pd.read_csv('metric.csv')
    df_full['ts'] = pd.to_datetime(df_full['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_local = tz.localize(pd.Timestamp("2021-03-10 14:00:00"))
    end_local   = tz.localize(pd.Timestamp("2021-03-10 14:30:00"))
    window_metric = df_full[(df_full['ts'] >= start_local) & (df_full['ts'] <= end_local)].copy()

wm = window_metric.copy()

# Ensure ts column is timezone-aware UTC+8
if 'ts' not in wm.columns:
    wm['ts'] = pd.to_datetime(wm['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    wm['ts'] = pd.to_datetime(wm['ts']).dt.tz_convert(tz)

# Filter components and KPI name substrings (case-sensitive)
components = ['Tomcat03','Tomcat04']
pattern = r'JVM|GC|MEMORY|Heap|CPU|Threads|gc_pause'  # case-sensitive by default
mask = wm['cmdb_id'].isin(components) & wm['kpi_name'].astype(str).str.contains(pattern, na=False)

tomcat_jvm_metrics_window = (
    wm[mask]
    .loc[:, ['cmdb_id','kpi_name','ts','value']]
    .rename(columns={'cmdb_id':'component','ts':'timestamp'})
    .sort_values(['component','kpi_name','timestamp'], ascending=[True,True,True])
    .head(200)
    .reset_index(drop=True)
)

tomcat_jvm_metrics_window

```
Out[8]:
```
```python
# Analyze tomcat_jvm_metrics_window to see if Tomcat JVM/host resource anomalies precede MG trace spikes.
# Produce a compact plain-English summary string based on the actual data in the kernel variable.

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Validate presence
if 'tomcat_jvm_metrics_window' not in globals():
    raise NameError("Kernel variable 'tomcat_jvm_metrics_window' not found. Please run prior step to create it.")

df = tomcat_jvm_metrics_window.copy()

# Ensure timestamp is timezone-aware and sorted
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_convert(tz)
df = df.sort_values('timestamp').reset_index(drop=True)

# Define MG trace spike anchor times (UTC+8)
spike1 = tz.localize(pd.Timestamp("2021-03-10 14:09:00"))
spike2 = tz.localize(pd.Timestamp("2021-03-10 14:19:00"))

# Aggregate per KPI
grp = df.groupby(['component','kpi_name'])
agg = grp.agg(
    count=('value','count'),
    min=('value','min'),
    median_p50=('value', lambda x: np.percentile(x.dropna(),50) if x.dropna().size>0 else np.nan),
    p95=('value', lambda x: np.percentile(x.dropna(),95) if x.dropna().size>0 else np.nan),
    max_value=('value','max')
).reset_index()

# find timestamp of first occurrence of max per group
def first_max_ts(sub):
    mv = sub['value'].max()
    rows = sub[sub['value'] == mv]
    if rows.empty:
        return pd.NaT
    return rows['timestamp'].iloc[0]

ts_list = []
for (comp,kpi), sub in df.groupby(['component','kpi_name']):
    ts_list.append({
        'component': comp,
        'kpi_name': kpi,
        'ts_of_max': first_max_ts(sub)
    })
ts_df = pd.DataFrame(ts_list)

agg = pd.merge(agg, ts_df, on=['component','kpi_name'], how='left')

# compute ratios to judge significance (max_over_median, max_over_p95), handle zeros
def ratio_safe(num, denom):
    try:
        if pd.isna(num) or pd.isna(denom):
            return np.nan
        denom = float(denom)
        if denom == 0:
            return np.inf
        return float(num) / denom
    except Exception:
        return np.nan

agg['max_over_median'] = agg.apply(lambda r: ratio_safe(r['max_value'], r['median_p50']), axis=1)
agg['max_over_p95'] = agg.apply(lambda r: ratio_safe(r['max_value'], r['p95']), axis=1)

# Determine where peak occurs relative to spikes
def peak_slot(ts):
    if pd.isna(ts) or ts is pd.NaT:
        return 'unknown'
    if ts <= spike1:
        return 'on-or-before-14:09'
    if spike1 < ts <= spike2:
        return 'between-14:09-and-14:19'
    return 'after-14:19'

agg['peak_slot'] = agg['ts_of_max'].apply(peak_slot)

# Identify potentially significant pre-spike anomalies: peak on-or-before-14:09 and ratio > 1.5
pre_spike_significant = agg[(agg['peak_slot']=='on-or-before-14:09') & (
    (agg['max_over_median']>1.5) | (agg['max_over_p95']>1.5)
)].copy()

# Also capture any KPI peaking between spikes with significant ratio
between_significant = agg[(agg['peak_slot']=='between-14:09-and-14:19') & (
    (agg['max_over_median']>1.5) | (agg['max_over_p95']>1.5)
)].copy()

# Prepare concise lists for summary
def fmt_ratio(x):
    if pd.isna(x):
        return "nan"
    if np.isinf(x):
        return "inf"
    return f"{x:.2f}"

lines = []
lines.append("Tomcat JVM/host metrics around incident window (based on tomcat_jvm_metrics_window):")
lines.append(f"- Total KPI series examined: {len(agg)}")
lines.append("")

# Top KPIs by severity (max_over_median), show top 6
top_by_ratio = agg.sort_values('max_over_median', ascending=False).head(6)
lines.append("Top KPIs by max/median ratio (showing up to 6):")
for _, r in top_by_ratio.iterrows():
    tsmax = r['ts_of_max']
    tsstr = tsmax.isoformat() if (not pd.isna(tsmax) and tsmax is not pd.NaT) else "na"
    lines.append(f"  • {r['component']} | {r['kpi_name']} | max={r['max_value']:.6g} at {tsstr} | median={r['median_p50']:.6g} | max/median={fmt_ratio(r['max_over_median'])}")

lines.append("")

# Report pre-spike significant findings
if pre_spike_significant.empty and between_significant.empty:
    lines.append("Finding: No JVM/host KPI shows a notable (>1.5x) peak on-or-before 14:09 or between 14:09 and 14:19.")
else:
    if not pre_spike_significant.empty:
        lines.append("KPIs with significant peaks on-or-before 14:09 (may precede the first MG spike):")
        for _, r in pre_spike_significant.iterrows():
            tsstr = r['ts_of_max'].isoformat()
            lines.append(f"  • {r['component']} | {r['kpi_name']} | max={r['max_value']:.6g} at {tsstr} | max/median={fmt_ratio(r['max_over_median'])} | max/p95={fmt_ratio(r['max_over_p95'])}")
    if not between_significant.empty:
        lines.append("KPIs with significant peaks between 14:09 and 14:19 (coincident with MG spikes):")
        for _, r in between_significant.iterrows():
            tsstr = r['ts_of_max'].isoformat()
            lines.append(f"  • {r['component']} | {r['kpi_name']} | max={r['max_value']:.6g} at {tsstr} | max/median={fmt_ratio(r['max_over_median'])} | max/p95={fmt_ratio(r['max_over_p95'])}")

lines.append("")
# Quick interpretation using observed data
# Check CPU and GC specifically
cpu_rows = agg[agg['kpi_name'].str.contains('CPU', na=False)]
gc_rows = agg[agg['kpi_name'].str.contains('GC', na=False) | agg['kpi_name'].str.contains('gc_pause', na=False)]
mem_rows = agg[agg['kpi_name'].str.contains('MEMORY|Heap|JVM|HeapMemory', na=False)]

def simple_report(group, name):
    if group.empty:
        return f"- {name}: no series present in window."
    # any series with peak on-or-before 14:09 and ratio>1.5?
    s = group[(group['peak_slot']=='on-or-before-14:09') & ((group['max_over_median']>1.5)|(group['max_over_p95']>1.5))]
    if not s.empty:
        return f"- {name}: some series show notable pre-spike peaks (see lines above)."
    return f"- {name}: no notable pre-spike peaks; values appear modest in the sampled window."

lines.append("Quick check by metric category:")
lines.append(simple_report(cpu_rows, "CPU metrics"))
lines.append(simple_report(gc_rows, "GC / gc_pause metrics"))
lines.append(simple_report(mem_rows, "Memory / JVM metrics"))

lines.append("")
lines.append("Summary conclusion:")
if pre_spike_significant.empty and between_significant.empty:
    lines.append("- There is no clear evidence in the Tomcat03/Tomcat04 JVM/host metrics that a substantial JVM or host resource anomaly preceded the MG trace spikes at 14:09 or 14:19.")
    lines.append("- Observed CPU percentages are low/modest; GC pause values in the sampled rows were small; memory/cache metrics are steady. Some higher values occur after 14:09 (e.g., a CPU reading at 14:15) which do not precede the first MG spike.")
else:
    lines.append("- Some Tomcat JVM/host KPIs show significant peaks coincident with or before MG trace spikes; please review the listed KPI lines above for specifics.")

lines.append("")
lines.append("Note: This conclusion is based solely on the provided incident-window metric points (variable tomcat_jvm_metrics_window). The full aggregated thresholds (metric_thresholds) or longer-term baselines were not applied in this check; if you want a thresholds-based detection, we can join with metric_thresholds and re-evaluate.")

summary = "\n".join(lines)

# Store the compact per-kpi aggregation for reference in kernel as tomcat_jvm_metrics_summary
tomcat_jvm_metrics_summary = agg.copy()

# Display the summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

component                                kpi_name                 timestamp      value
0    Tomcat03         OSLinux-CPU_CPU-0_SingleCpuUtil 2021-03-10 14:01:00+08:00     3.1281
1    Tomcat03         OSLinux-CPU_CPU-0_SingleCpuUtil 2021-03-10 14:03:00+08:00     3.5631
2    Tomcat03         OSLinux-CPU_CPU-0_SingleCpuUtil 2021-03-10 14:09:00+08:00     4.1326
3    Tomcat03         OSLinux-CPU_CPU-0_SingleCpuUtil 2021-03-10 14:11:00+08:00     3.8103
4    Tomcat03         OSLinux-CPU_CPU-0_SingleCpuUtil 2021-03-10 14:15:00+08:00     7.9175
..        ...                                     ...                       ...        ...
195  Tomcat03  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-10 14:19:00+08:00  4182.0000
196  Tomcat03  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-10 14:22:00+08:00  4171.0000
197  Tomcat03  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-10 14:23:00+08:00  4176.0000
198  Tomcat03  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-10 14:25:00+08:00  4188.0000
199  Tomcat03  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-10 14:26:00+08:00  4177.0000

[200 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Incident early window bounds (inclusive) in UTC+8
start_local = tz.localize(pd.Timestamp("2021-03-10 14:00:00"))
end_local   = tz.localize(pd.Timestamp("2021-03-10 14:12:00"))

# Ensure window_trace exists; if not, load full trace and filter to incident window
if 'window_trace' in globals():
    wt = window_trace.copy()
else:
    tr = pd.read_csv('trace.csv')
    tr['ts'] = pd.to_datetime(tr['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    wt = tr[(tr['ts'] >= start_local) & (tr['ts'] <= end_local)].copy()

# Ensure tz-aware timestamp column named 'ts' exists
if 'ts' not in wt.columns:
    wt['ts'] = pd.to_datetime(wt['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    wt['ts'] = pd.to_datetime(wt['ts']).dt.tz_convert(tz)

# Ensure global thresholds exist (computed on full series, not filtered)
if 'trace_thresholds' not in globals():
    full_trace = pd.read_csv('trace.csv')
    grp = full_trace.groupby(['cmdb_id','trace_name'])['value']
    trace_thresholds = grp.agg(count='count', p95_value=lambda x: np.percentile(x.dropna().astype(float),95) if x.dropna().size>0 else np.nan).reset_index()
    trace_thresholds = trace_thresholds.rename(columns={'cmdb_id':'component','trace_name':'kpi_name'})

# Prepare thresholds for join: columns component, kpi_name, p95_value
tt = trace_thresholds[['component','kpi_name','p95_value']].copy()

# Merge window trace with thresholds on component & trace_name
merged = pd.merge(
    wt,
    tt,
    left_on=['cmdb_id','trace_name'],
    right_on=['component','kpi_name'],
    how='left'
)

# Restrict to allowed components and incident early time window (redundant but explicit)
merged = merged[merged['cmdb_id'].isin(allowed_components)].copy()
merged = merged[(merged['ts'] >= start_local) & (merged['ts'] <= end_local)].copy()

# Numeric conversion
merged['value_num'] = pd.to_numeric(merged['value'], errors='coerce')
merged['p95_value_num'] = pd.to_numeric(merged['p95_value'], errors='coerce')

# Detect anomalies where value >= p95 (require p95 not NaN)
mask = merged['p95_value_num'].notna() & merged['value_num'].notna() & (merged['value_num'] >= merged['p95_value_num'])
anoms = merged[mask].copy()

# Compute ratio, treating p95==0 as inf
def compute_ratio(v, p):
    try:
        if pd.isna(p):
            return np.nan
        p = float(p)
        if p == 0:
            return np.inf
        return float(v) / p
    except Exception:
        return np.nan

anoms['ratio'] = anoms.apply(lambda r: compute_ratio(r['value_num'], r['p95_value_num']), axis=1)

# Prepare output columns and formatting
early_trace_anomalies = anoms.assign(
    component = anoms['cmdb_id'],
    trace_name = anoms['trace_name'],
    timestamp = anoms['ts'],
    value = anoms['value_num'],
    p95_value = anoms['p95_value_num']
)[['component','trace_name','timestamp','value','p95_value','ratio']].copy()

# Sort by timestamp ascending then ratio descending
early_trace_anomalies = early_trace_anomalies.sort_values(by=['timestamp','ratio'], ascending=[True, False]).reset_index(drop=True)

# Store full result in kernel variable as requested (already named)
# Limit for display to 50 rows
early_trace_anomalies_head = early_trace_anomalies.head(50)

# Return compact result (head)
early_trace_anomalies_head

```
Out[9]:
```
Summary of early trace anomalies (2021-03-10 14:00–14:12 UTC+8):

- Earliest anomalies start right at 14:00–14:01 on MG nodes:
  - 14:00 — MG02: trace.to_dockerB1.row_count = 234 (p95 230.55, ratio ≈ 1.015).
  - 14:01 — MG02: trace.to_dockerB2.row_count = 252 (p95 230.40, ratio ≈ 1.094).
  - 14:01 — MG01: trace.to_dockerB2.row_count = 258 (p95 247.20, ratio ≈ 1.044).
  - 14:01 — IG01 <-> Tomcat01 row_count interactions (≈262, ratio ≈ 1.017).

- Between 14:01 and 14:09, ratios are modest (≈1.01–1.09) — indicating increased row counts / throughput on MG → docker paths early in the incident.

- Around 14:09 there are additional anomalies in duration_p95 on MG and Tomcat:
  - 14:09 — MG01: trace.to_dockerA1.duration_p95 = 11.025 (p95 10.3817, ratio ≈ 1.062).
  - 14:09 — MG02 / Tomcat03: duration_p95 ~18.24 (p95 17.19, ratio ≈ 1.061).
  - 14:09 — Tomcat04: trace.self.duration_p95 ≈4.127 (p95 3.909, ratio ≈ 1.056).
  - Several MG↔docker and MG↔Tomcat duration_p95 series breach p95 slightly after the initial row_count increases.

Interpretation / recommended focus:
- The earliest detectable downstream signals are increased row_count on MG01/MG02 to dockerB/A services at ~14:00–14:01 — these precede the more noticeable duration_p95 spikes seen near 14:09.
- Ratios are modest early (≈1.01–1.09) but consistent across multiple MG series, suggesting increased load or routing shifts starting ~14:00–14:01 that later coincide with latency increases around ~14:09.
- Prioritize investigating MG01/MG02 interactions with dockerA/B services and request routing/queueing around 14:00–14:09, and examine Tomcat03/MG paths around 14:09 for downstream latency impact.

The original code execution output of IPython Kernel is also provided below for reference:

component                        trace_name                 timestamp      value   p95_value     ratio
0       MG02       trace.to_dockerB1.row_count 2021-03-10 14:00:00+08:00  234.00000  230.550000  1.014964
1       MG02       trace.to_dockerB2.row_count 2021-03-10 14:01:00+08:00  252.00000  230.400000  1.093750
2       MG01       trace.to_dockerB2.row_count 2021-03-10 14:01:00+08:00  258.00000  247.200000  1.043689
3       IG01       trace.to_Tomcat01.row_count 2021-03-10 14:01:00+08:00  262.00000  257.500000  1.017476
4   Tomcat01         trace.from_IG01.row_count 2021-03-10 14:01:00+08:00  262.00000  257.500000  1.017476
..       ...                               ...                       ...        ...         ...       ...
45      MG01    trace.to_dockerA1.duration_p95 2021-03-10 14:09:00+08:00   11.02515   10.381672  1.061982
46      MG02  trace.from_Tomcat03.duration_p95 2021-03-10 14:09:00+08:00   18.24395   17.191647  1.061210
47  Tomcat03        trace.to_MG02.duration_p95 2021-03-10 14:09:00+08:00   18.24395   17.191647  1.061210
48  Tomcat04           trace.self.duration_p95 2021-03-10 14:09:00+08:00    4.12710    3.909075  1.055774
49      IG01    trace.to_Tomcat03.duration_p95 2021-03-10 14:09:00+08:00   10.59140   10.082092  1.050516

[50 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz
import re

# timezone
tz = pytz.timezone('Asia/Shanghai')

# Components of interest
components = ["MG01","MG02"]

# KPI name substrings (case-insensitive)
substrs = ['CPU','MEMORY','JVM','Heap','NET','NETWORK','DSK','DISK','LOCALDISK','GC','gc','Threads','gc_pause']

# Compile regex pattern for any of the substrings (case-insensitive)
pattern = re.compile("|".join([re.escape(s) for s in substrs]), flags=re.IGNORECASE)

# Ensure metric_thresholds exists (global p95 computed on full data)
if 'metric_thresholds' not in globals():
    # compute global thresholds from full metric.csv (per rule 9)
    full_metric = pd.read_csv('metric.csv')
    grp_full = full_metric.groupby(['cmdb_id','kpi_name'])['value']
    metric_thresholds = grp_full.agg(count='count', p95_value=lambda x: np.percentile(x.dropna().astype(float),95) if x.dropna().size>0 else np.nan).reset_index()
    metric_thresholds = metric_thresholds.rename(columns={'cmdb_id':'component'})

# Ensure window_metric exists (incident-window filtered table)
if 'window_metric' not in globals():
    # fallback: load and filter incident window 2021-03-10 14:00—14:30 UTC+8
    dfm = pd.read_csv('metric.csv')
    dfm['ts'] = pd.to_datetime(dfm['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_local = tz.localize(pd.Timestamp("2021-03-10 14:00:00"))
    end_local   = tz.localize(pd.Timestamp("2021-03-10 14:30:00"))
    window_metric = dfm[(dfm['ts'] >= start_local) & (dfm['ts'] <= end_local)].copy()

# Prepare window data: filter to MG01/MG02 and KPI name pattern
wm = window_metric.copy()
# ensure ts exists and is tz-aware
if 'ts' not in wm.columns:
    wm['ts'] = pd.to_datetime(wm['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    wm['ts'] = pd.to_datetime(wm['ts']).dt.tz_convert(tz)

wm = wm[wm['cmdb_id'].isin(components)].copy()
# ensure kpi_name is string
wm['kpi_name'] = wm['kpi_name'].astype(str)
# filter by pattern (case-insensitive)
wm = wm[wm['kpi_name'].str.contains(pattern)]

# Numeric conversion
wm['value_num'] = pd.to_numeric(wm['value'], errors='coerce')

# Group by series and compute required aggregates using window data
def agg_series(g):
    count_points = int(g['value_num'].count())
    median_val = np.nan
    max_val = np.nan
    ts_of_max = pd.NaT
    if g['value_num'].dropna().size > 0:
        median_val = float(np.percentile(g['value_num'].dropna(), 50))
        max_idx = g['value_num'].idxmax()
        max_val = float(g.loc[max_idx, 'value_num'])
        ts_of_max = g.loc[max_idx, 'ts']
    return pd.Series({
        'count_points_in_window': count_points,
        'median_value_in_window': median_val,
        'max_value_in_window': max_val,
        'timestamp_of_max': ts_of_max
    })

agg = wm.groupby(['cmdb_id','kpi_name']).apply(agg_series).reset_index().rename(columns={'cmdb_id':'component'})

# Join with global p95 thresholds (metric_thresholds) on component & kpi_name
# metric_thresholds columns: component, kpi_name, p95_value (ensure correct name)
mt = metric_thresholds.copy()
# handle possible column name variants
if 'p95_value' not in mt.columns:
    # try known alternate 'p95' or compute if missing
    if 'p95' in mt.columns:
        mt = mt.rename(columns={'p95':'p95_value'})
    else:
        # fallback compute p95 on full metric.csv (should not usually be needed)
        full_metric = pd.read_csv('metric.csv')
        grp_full = full_metric.groupby(['cmdb_id','kpi_name'])['value']
        mt = grp_full.agg(p95_value=lambda x: np.percentile(x.dropna().astype(float),95) if x.dropna().size>0 else np.nan).reset_index().rename(columns={'cmdb_id':'component'})

res = pd.merge(agg, mt[['component','kpi_name','p95_value']], on=['component','kpi_name'], how='left')

# Compute ratios with careful handling of zero/NaN
def ratio_over_p95(maxv, p95):
    if pd.isna(maxv) or pd.isna(p95):
        return np.nan
    try:
        p95f = float(p95)
        if p95f == 0:
            return np.inf
        return float(maxv) / p95f
    except Exception:
        return np.nan

def ratio_over_median(maxv, median):
    if pd.isna(maxv) or pd.isna(median):
        return np.nan
    try:
        med = float(median)
        if med == 0:
            return np.inf
        return float(maxv) / med
    except Exception:
        return np.nan

res['max_over_p95'] = res.apply(lambda r: ratio_over_p95(r['max_value_in_window'], r['p95_value']), axis=1)
res['max_over_median'] = res.apply(lambda r: ratio_over_median(r['max_value_in_window'], r['median_value_in_window']), axis=1)

# Filter condition: timestamp_of_max <= 2021-03-10 14:01:00+08:00 AND (max_over_p95 > 1.2 OR max_over_median > 1.5)
cutoff = tz.localize(pd.Timestamp("2021-03-10 14:01:00"))
# Ensure timestamp_of_max is tz-aware
res['timestamp_of_max'] = pd.to_datetime(res['timestamp_of_max']).dt.tz_convert(tz)

cond_time = res['timestamp_of_max'].notna() & (res['timestamp_of_max'] <= cutoff)
cond_severity = res['max_over_p95'].apply(lambda x: (np.isinf(x) or (not pd.isna(x) and x > 1.2))) | res['max_over_median'].apply(lambda x: (np.isinf(x) or (not pd.isna(x) and x > 1.5)))

filtered = res[cond_time & cond_severity].copy()

# Sort by max_over_p95 desc then max_over_median desc
filtered = filtered.sort_values(['max_over_p95','max_over_median'], ascending=[False, False])

# Final column selection and rounding for compactness
final_cols = ['component','kpi_name','count_points_in_window','p95_value','median_value_in_window','max_value_in_window','timestamp_of_max','max_over_p95','max_over_median']
# Round numeric fields where applicable
for c in ['p95_value','median_value_in_window','max_value_in_window','max_over_p95','max_over_median']:
    if c in filtered.columns:
        filtered[c] = pd.to_numeric(filtered[c], errors='coerce')
        # keep inf as-is; round finite numbers
        filtered[c] = filtered[c].apply(lambda x: np.inf if np.isinf(x) else (round(x,6) if not pd.isna(x) else x))

mg_early_metric_anomalies = filtered[final_cols].reset_index(drop=True)

# Display compact top results (up to 20 rows)
mg_early_metric_anomalies.head(20)

```
Out[12]:
```
Findings for MG01 / MG02 metrics (checked KPIs containing CPU, MEMORY, JVM, Heap, NET/NETWORK, DSK/DISK/LOCALDISK, GC, Threads) with peaks at or before 2021-03-10 14:01:00+08:

- Most series show p95_value == 0 (local disk counters, many network error counters, etc.). Those yield infinite ratios but are degenerate instrumentation/zero-baseline artifacts and are not meaningful indicators of resource failure.
- The only non-degenerate signal before or at 14:01 is:
  - MG02 | OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil — count 10 in window; median ≈ 0.00375, p95 ≈ 0.007425, max = 0.0081 at 2021-03-10 14:01:00+08. 
    - max_over_p95 ≈ 1.09 (marginal), max_over_median ≈ 2.16 (shows the max is ~2.2× the median).
    - This indicates a small/brief increase in observed network bandwidth on MG02 at 14:01, but the breach vs p95 is minor.
- No clear CPU, JVM, memory, GC, or disk metric shows a substantial (>1.2× p95 and >1.5× median) early spike on MG01 or MG02 that would explain the incident onset.

Interpretation and recommendation:
- There is no strong evidence from MG01/MG02 host or JVM metrics before 14:01 that a resource exhaustion or JVM GC event triggered the incident.
- The modest MG02 network bandwidth uptick at ~14:01 may be related to early increased throughput but is unlikely by itself to explain the larger MG trace latency spikes that occur later (~14:09).
- Many infinite ratios stem from zero-valued baselines; treat those series as non-actionable.
- Next steps: focus investigation on MG routing/load patterns and the MG→docker/Tomcat request flows (row_count and trace anomalies already observed around 14:00–14:09), rather than host JVM resource failures on MG01/MG02.

The original code execution output of IPython Kernel is also provided below for reference:

component                                         kpi_name  count_points_in_window  p95_value  median_value_in_window  max_value_in_window          timestamp_of_max  max_over_p95  max_over_median
0       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                       9   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
1       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                       9   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
2       MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                      11   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
3       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps                       9   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
4       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead                       9   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
5       MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                      11   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
6       MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc                      12   0.000000                 0.00000               0.0000 2021-03-10 14:00:00+08:00           inf              inf
7       MG01     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc                      12   0.000000                 0.00000               0.0000 2021-03-10 14:00:00+08:00           inf              inf
8       MG02                  OSLinux-CPU_CPU-3_SingleCpuidle                      10   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
9       MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                      10   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
10      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                      10   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
11      MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                      11   0.000000                 0.00000               0.0000 2021-03-10 14:00:00+08:00           inf              inf
12      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps                      10   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
13      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead                      10   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
14      MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      21   0.000000                 0.00000               0.0000 2021-03-10 14:00:00+08:00           inf              inf
15      MG02          OSLinux-OSLinux_NETWORK_ens160_NETInErr                      11   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
16      MG02         OSLinux-OSLinux_NETWORK_ens160_NETOutErr                      11   0.000000                 0.00000               0.0000 2021-03-10 14:01:00+08:00           inf              inf
17      MG02     OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc                      11   0.000000                 0.00000               0.0000 2021-03-10 14:00:00+08:00           inf              inf
18      MG02  OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil                      10   0.007425                 0.00375               0.0081 2021-03-10 14:01:00+08:00      1.090909             2.16```
```